# Klein style steering — origin vs steered

Left: T2I **without** steering. Right: T2I **with** steering. The reference image used to compute the vector is shown above the pairs.

Captions include `strength` (txt stream) and `strength_img` parsed from steered filenames (`_s_{strength}_simg_{strength_img}_`).

Set `EXP_NAME` to the image stem (folder under `experiments/klein_9b/style/`), then run all cells.

```bash
bash scripts/run_klein_style.sh data/reference_images/vangogh.jpg
bash scripts/apply_steering_klein.sh vangogh
```

In [ ]:
%matplotlib inline

from pathlib import Path
import re

from IPython.display import display, Markdown
from PIL import Image
import matplotlib.pyplot as plt

# Folder name = reference image stem, e.g. vangogh.jpg → vangogh
EXP_NAME = "vangogh"

# Optional: only show this txt-stream strength (filename _s_{value}_). None = all.
STRENGTH = None

# Optional: limit how many prompt pairs to plot. None = all.
MAX_PAIRS = None

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
STRENGTH_RE = re.compile(
    r"_s_(?P<strength>[-+]?\d+(?:\.\d+)?)_simg_(?P<strength_img>[-+]?\d+(?:\.\d+)?)_v_(?P<vector_type>[^.]+)",
    re.IGNORECASE,
)

In [ ]:
def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "steering" / "apply_steering_klein.py").is_file():
            return candidate
    return here


def list_images(directory: Path):
    if not directory.is_dir():
        return []
    files = [p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    files.sort(key=lambda p: p.name)
    return files


def prompt_index(path: Path):
    match = re.match(r"^(\d+)_", path.stem)
    return int(match.group(1)) if match else None


def prompt_text(path: Path) -> str:
    stem = path.stem
    stem = re.sub(r"^\d+_", "", stem)
    stem = re.sub(r"_origin$", "", stem)
    stem = STRENGTH_RE.sub("", stem)
    return stem.replace("_", " ").strip()


def parse_strength(path: Path):
    match = STRENGTH_RE.search(path.stem)
    if not match:
        return None
    return {
        "strength": float(match.group("strength")),
        "strength_img": float(match.group("strength_img")),
        "vector_type": match.group("vector_type"),
    }


def strength_caption(info) -> str:
    if not info:
        return "strength: (not in filename)"
    return (
        f"strength={info['strength']:g}, "
        f"strength_img={info['strength_img']:g}, "
        f"vector={info['vector_type']}"
    )


def find_reference(exp_root: Path) -> Path:
    matches = sorted(
        p for p in exp_root.iterdir()
        if p.is_file() and p.stem.lower().startswith("reference") and p.suffix.lower() in IMAGE_EXTS
    )
    if matches:
        return matches[0]
    raise FileNotFoundError(
        f"No reference.* image in {exp_root}. Re-run get_vector_klein.sh so it copies the ref image."
    )


ROOT = find_repo_root()
EXP_ROOT = ROOT / "experiments" / "klein_9b" / "style" / EXP_NAME
ORIGIN_DIR = EXP_ROOT / "generated_images" / "origin"
STEERED_DIR = EXP_ROOT / "generated_images" / "steered"

print(f"repo:     {ROOT}")
print(f"exp:      {EXP_ROOT}")
print(f"origin:   {ORIGIN_DIR}  exists={ORIGIN_DIR.is_dir()}")
print(f"steered:  {STEERED_DIR}  exists={STEERED_DIR.is_dir()}")

In [ ]:
if not EXP_ROOT.is_dir():
    available = sorted(
        p.name for p in (ROOT / "experiments" / "klein_9b" / "style").glob("*")
        if p.is_dir()
    )
    raise FileNotFoundError(
        f"Experiment folder not found: {EXP_ROOT}\n"
        f"Available: {available or '(none)'}\n"
        "Set EXP_NAME to the reference-image stem."
    )

REF_PATH = find_reference(EXP_ROOT)
origin_files = {prompt_index(p): p for p in list_images(ORIGIN_DIR) if prompt_index(p) is not None}
steered_files = list_images(STEERED_DIR)
if STRENGTH is not None:
    token = f"_s_{float(STRENGTH)}_"
    steered_files = [p for p in steered_files if token in p.name]

pairs = []
missing_origin = []
for steered in steered_files:
    idx = prompt_index(steered)
    if idx is None:
        continue
    origin = origin_files.get(idx)
    if origin is None:
        missing_origin.append(idx)
        continue
    pairs.append((idx, origin, steered, parse_strength(steered), prompt_text(origin)))

pairs.sort(key=lambda row: row[0])
if MAX_PAIRS is not None:
    pairs = pairs[:MAX_PAIRS]

if missing_origin:
    print(f"Steered images with no origin match (re-run apply with --save_origin): {missing_origin}")
if not pairs:
    raise FileNotFoundError(
        f"No origin/steered pairs in {EXP_ROOT / 'generated_images'}. "
        "Run: bash scripts/apply_steering_klein.sh " + EXP_NAME
    )

print(f"reference: {REF_PATH}")
print(f"pairs:     {len(pairs)}")
for idx, origin, steered, info, prompt in pairs[:5]:
    print(f"  {idx:02d}  {prompt[:60]!r}  {strength_caption(info)}")

## Reference image

Style image used on the positive extract pass (the vector is `mean(pos) − mean(neg)`).

In [ ]:
ref = Image.open(REF_PATH).convert("RGB")
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(ref)
ax.set_title(f"Reference ({EXP_NAME})\n{REF_PATH.name}", fontsize=11)
ax.axis("off")
fig.tight_layout()
plt.show()

## Origin (left) vs steered (right)

Each row is one prompt. Caption lists txt-stream `strength` and image-stream `strength_img`.

In [ ]:
n = len(pairs)
fig, axes = plt.subplots(n, 2, figsize=(10, 4.8 * n), squeeze=False)

for row, (idx, origin_path, steered_path, info, prompt) in enumerate(pairs):
    cap = strength_caption(info)
    label = prompt if prompt else f"prompt {idx:02d}"
    if len(label) > 70:
        label = label[:67] + "..."

    left, right = axes[row]
    left.imshow(Image.open(origin_path).convert("RGB"))
    left.set_title(f"#{idx:02d}  without steering\n{label}\n{cap}", fontsize=9)
    left.set_xticks([])
    left.set_yticks([])

    right.imshow(Image.open(steered_path).convert("RGB"))
    right.set_title(f"#{idx:02d}  with steering\n{label}\n{cap}", fontsize=9)
    right.set_xticks([])
    right.set_yticks([])

fig.suptitle(
    f"{EXP_NAME}: origin (left) vs steered (right)\nreference: {REF_PATH.name}",
    fontsize=12,
)
fig.tight_layout()
plt.show()

## Compact strip (reference + one pair)

Useful for a paper-style figure: reference | unsteered | steered.

In [ ]:
PAIR_INDEX = 0  # which prompt pair from `pairs`
SAVE_PATH = EXP_ROOT / "generated_images" / f"compare_{EXP_NAME}.png"  # set None to skip save

idx, origin_path, steered_path, info, prompt = pairs[PAIR_INDEX]
cap = strength_caption(info)
label = prompt if prompt else f"prompt {idx:02d}"

fig, axes = plt.subplots(1, 3, figsize=(14, 5.2))
panels = [
    (REF_PATH, f"Reference\n{EXP_NAME}"),
    (origin_path, f"Without steering\n{label}"),
    (steered_path, f"With steering\n{label}"),
]
for ax, (path, title) in zip(axes, panels):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(f"{title}\n{cap}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(cap, fontsize=12)
fig.tight_layout()
if SAVE_PATH is not None:
    SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(SAVE_PATH, dpi=150, bbox_inches="tight")
    print(f"saved {SAVE_PATH}")
plt.show()